# 03 Anomaly Analysis

Detect unusual weather records using z-scores and Isolation Forest, then save anomaly-tagged and anomaly-free outputs.


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "GlobalWeatherRepository.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.preprocessing import (
    add_isolation_forest_anomalies,
    compute_zscore_outliers,
    get_numeric_columns,
    load_processed_data,
    remove_anomalies,
    save_processed_data,
)
from src.visualize import (
    plot_kde_comparison,
    plot_scatter_with_hue,
    plot_top_categories,
    save_figure,
)


In [ ]:
clean_data_path = PROCESSED_DIR / "clean_weather_data.csv"
df = load_processed_data(clean_data_path)
numeric_columns = get_numeric_columns(df)

df["zscore_outlier"] = compute_zscore_outliers(df, numeric_columns)
anomaly_df = add_isolation_forest_anomalies(df, numeric_columns, contamination=0.05)
anomaly_df["zscore_outlier"] = df["zscore_outlier"]

print(anomaly_df[["zscore_outlier", "anomaly"]].value_counts())
display(anomaly_df.head())


In [ ]:
plt.figure(figsize=(16, 6))
anomaly_df[numeric_columns].plot(kind="box", rot=90)
plt.title("Boxplots of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
fig_scatter = plot_scatter_with_hue(
    anomaly_df,
    x="temperature_celsius",
    y="humidity",
    hue="anomaly",
    title="Temperature vs Humidity by Isolation Forest Label",
)
display(fig_scatter)
save_figure(fig_scatter, FIGURES_DIR / "anomaly_temperature_humidity.png")
plt.close(fig_scatter)

fig_temp = plot_kde_comparison(
    anomaly_df,
    value_column="temperature_celsius",
    group_column="anomaly",
    label_map={1: "Normal", -1: "Anomaly"},
    title="Temperature Distribution: Normal vs Anomaly",
)
display(fig_temp)
plt.close(fig_temp)

fig_country = plot_top_categories(
    anomaly_df.loc[anomaly_df["anomaly"] == -1, "country"],
    top_n=10,
    title="Top 10 Countries with Most Anomalies",
    color="firebrick",
)
display(fig_country)
save_figure(fig_country, FIGURES_DIR / "anomaly_top_countries.png")
plt.close(fig_country)


In [ ]:
anomaly_output = PROCESSED_DIR / "anomaly_scored_weather_data.csv"
normal_output = PROCESSED_DIR / "weather_without_anomalies.csv"

normal_df = remove_anomalies(anomaly_df)
save_processed_data(anomaly_df, anomaly_output)
save_processed_data(normal_df, normal_output)

print(f"Saved anomaly-scored data to: {anomaly_output}")
print(f"Saved anomaly-free data to: {normal_output}")
print(f"Anomaly-free shape: {normal_df.shape}")
